In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.layers import Dense, Activation, Dropout
import time #helper libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from keras.layers import Dense, Activation, Dropout
import time #helper libraries
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
'''
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
'''
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

ModuleNotFoundError: No module named 'keras'

# Cleaning up the df files an merging

In [24]:
df_inds= pd.read_csv("/kaggle/input/cdl-fall25-textual/Brianferrell_analyzed_2023.csv")

In [25]:
df_stocks = pd.read_csv("/kaggle/input/cdl-fall25-textual/huggingface_mdnh_hourly-stock-data-2023.csv")

In [26]:
df_inds["scores"] = df_inds["processed"].str.findall(r"-?\d+")


In [27]:
df_inds["processed"]

0                      2 3 0 4 1 5 2 3 0 0
1           -1\n4\n5\n0\n5\n2\n-1\n1\n0\n3
2          -2\n0\n5\n-3\n4\n-1\n0\n1\n0\n3
3          -3\n2\n0\n1\n-1\n0\n0\n-4\n0\n5
4                     3 -1 2 5 4 0 1 2 5 6
                       ...                
24308       -1\n0\n2\n4\n3\n1\n0\n-1\n0\n1
24309               3 0 2 -1 1 -4 5 4 -2 2
24310      -2\n3\n1\n4\n0\n-1\n5\n-3\n1\n0
24311    -3\n-1\n2\n0\n5\n-2\n-1\n0\n-1\n1
24312      -5\n1\n4\n0\n3\n2\n-1\n1\n-4\n3
Name: processed, Length: 24313, dtype: object

In [28]:
df_inds["scores"]

0             [2, 3, 0, 4, 1, 5, 2, 3, 0, 0]
1           [-1, 4, 5, 0, 5, 2, -1, 1, 0, 3]
2          [-2, 0, 5, -3, 4, -1, 0, 1, 0, 3]
3          [-3, 2, 0, 1, -1, 0, 0, -4, 0, 5]
4            [3, -1, 2, 5, 4, 0, 1, 2, 5, 6]
                        ...                 
24308       [-1, 0, 2, 4, 3, 1, 0, -1, 0, 1]
24309      [3, 0, 2, -1, 1, -4, 5, 4, -2, 2]
24310      [-2, 3, 1, 4, 0, -1, 5, -3, 1, 0]
24311    [-3, -1, 2, 0, 5, -2, -1, 0, -1, 1]
24312      [-5, 1, 4, 0, 3, 2, -1, 1, -4, 3]
Name: scores, Length: 24313, dtype: object

In [29]:
df_inds.columns
df_inds.head()


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,date,text,Impact Scores,processed,scores
0,2023-01-01 06:00:00+00:00,Credefi Scores Major Milestone in Partnership ...,NaN,2 3 0 4 1 5 2 3 0 0,"[2, 3, 0, 4, 1, 5, 2, 3, 0, 0]"
1,2023-01-01 09:00:00+00:00,"NIO Inc. Provides December, Fourth Quarter and...",NaN,-1\n4\n5\n0\n5\n2\n-1\n1\n0\n3,"[-1, 4, 5, 0, 5, 2, -1, 1, 0, 3]"
2,2023-01-01 10:25:00+00:00,CGTN: CMG president sends New Year greetings t...,NaN,-2\n0\n5\n-3\n4\n-1\n0\n1\n0\n3,"[-2, 0, 5, -3, 4, -1, 0, 1, 0, 3]"
3,2023-01-01 14:54:04+00:00,Toyota's Indian unit warns of a possible custo...,NaN,-3\n2\n0\n1\n-1\n0\n0\n-4\n0\n5,"[-3, 2, 0, 1, -1, 0, 0, -4, 0, 5]"
4,2023-01-01 15:00:00+00:00,"ZEASN Appoints Dr. Jack Q. Gao, Former Preside...",NaN,3 -1 2 5 4 0 1 2 5 6,"[3, -1, 2, 5, 4, 0, 1, 2, 5, 6]"


In [30]:
df_stocks.columns
df_stocks.head()
#df_stocks.size

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Unnamed: 0,timestamp,symbol,open,high,low,close,volume,sma_10,sma_50,ema_20,rsi_14,macd,macd_signal,macd_hist,volatility_20,target_up_next
0,0,2023-01-03 09:00:00,AAPL,130.28,131.46,130.28,131.30,143683,131.300000,131.300000,131.300000,0.000000,0.000000,0.000000,0.000000,NaN,0
1,1,2023-01-03 10:00:00,AAPL,131.40,131.40,130.80,131.00,42853,131.150000,131.150000,131.271429,0.000000,-0.023932,-0.004786,-0.019145,NaN,0
2,2,2023-01-03 11:00:00,AAPL,130.91,131.25,130.05,130.38,84001,130.893333,130.893333,131.186531,0.000000,-0.091867,-0.022203,-0.069665,0.007741,1
3,3,2023-01-03 12:00:00,AAPL,130.38,130.75,130.03,130.50,184600,130.795000,130.795000,131.121147,11.538462,-0.134474,-0.044657,-0.089817,0.012679,1
4,4,2023-01-03 13:00:00,AAPL,130.25,131.23,129.63,130.92,504693,130.820000,130.820000,131.101990,36.986301,-0.132818,-0.062289,-0.070529,0.015653,0


In [48]:
df_stocks.drop(columns=["Unnamed: 0"], inplace=True)

In [49]:
df_stocks.head()


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,timestamp,symbol,open,high,low,close,volume,sma_10,sma_50,ema_20,rsi_14,macd,macd_signal,macd_hist,volatility_20,target_up_next,day,hour
8301,2023-01-03 09:00:00,TSLA,120.02,121.23,115.94,118.52,1306706,118.5200,118.5200,118.520000,0.000000,0.000000,0.000000,0.000000,NaN,1,2023-01-03,9
8302,2023-01-03 10:00:00,TSLA,118.53,120.50,118.20,120.38,568635,119.4500,119.4500,118.697143,0.000000,0.148376,0.029675,0.118701,NaN,0,2023-01-03,10
8303,2023-01-03 11:00:00,TSLA,120.28,120.50,117.95,118.61,547403,119.1700,119.1700,118.688844,51.239669,0.121738,0.048088,0.073650,0.096124,0,2023-01-03,11
8304,2023-01-03 12:00:00,TSLA,118.63,118.99,116.61,118.28,1529445,118.9475,118.9475,118.649906,46.969697,0.073155,0.053101,0.020054,0.068494,1,2023-01-03,12
8305,2023-01-03 13:00:00,TSLA,118.32,121.90,116.61,118.93,2886437,118.9440,118.9440,118.676582,54.446855,0.086110,0.059703,0.026407,0.057561,0,2023-01-03,13


In [32]:
df_stocks = df_stocks[df_stocks["symbol"]=="TSLA"]

In [33]:
df_inds.drop(columns=["text", "Impact Scores", "processed"], inplace=True)

In [34]:
df_inds.head()

,date,scores
0,2023-01-01 06:00:00+00:00,"[2, 3, 0, 4, 1, 5, 2, 3, 0, 0]"
1,2023-01-01 09:00:00+00:00,"[-1, 4, 5, 0, 5, 2, -1, 1, 0, 3]"
2,2023-01-01 10:25:00+00:00,"[-2, 0, 5, -3, 4, -1, 0, 1, 0, 3]"
3,2023-01-01 14:54:04+00:00,"[-3, 2, 0, 1, -1, 0, 0, -4, 0, 5]"
4,2023-01-01 15:00:00+00:00,"[3, -1, 2, 5, 4, 0, 1, 2, 5, 6]"


In [35]:
df_stocks.size

17629

## merge

In [36]:
df_stocks["timestamp"] = pd.to_datetime(df_stocks["timestamp"])


In [37]:
df_stocks["timestamp"]

8301   2023-01-03 09:00:00
8302   2023-01-03 10:00:00
8303   2023-01-03 11:00:00
8304   2023-01-03 12:00:00
8305   2023-01-03 13:00:00
               ...        
9333   2023-12-08 13:00:00
9334   2023-12-08 14:00:00
9335   2023-12-08 15:00:00
9336   2023-12-08 16:00:00
9337   2023-12-08 17:00:00
Name: timestamp, Length: 1037, dtype: datetime64[ns]

In [39]:
df_inds["date"] = pd.to_datetime(df_inds["date"])

In [40]:
df_stocks["day"] = df_stocks["timestamp"].dt.date
df_stocks["hour"] = df_stocks["timestamp"].dt.hour

df_inds["day"] = df_inds["date"].dt.date
df_inds["hour"] = df_inds["date"].dt.hour


In [44]:
df_inds["hour"]

0         6
1         9
2        10
3        14
4        15
         ..
24308    16
24309    17
24310    17
24311    17
24312    18
Name: hour, Length: 24313, dtype: int32

In [50]:
df_inds["seq"] = df_inds.groupby(["day", "hour"]).cumcount()

In [51]:
df_inds["seq"]

0        0
1        0
2        0
3        0
4        0
        ..
24308    0
24309    0
24310    1
24311    2
24312    0
Name: seq, Length: 24313, dtype: int64

In [58]:
df_inds_lists = (
    df_inds
    .sort_values(["day", "hour"])                       # preserve original order if needed
    .groupby(["day", "hour"], sort=False)["scores"]
    .apply(list)
    .reset_index(name="scores_list")
)

In [59]:
df_inds_lists

,day,hour,scores_list
0,2023-01-01,6,"[[2, 3, 0, 4, 1, 5, 2, 3, 0, 0]]"
1,2023-01-01,9,"[[-1, 4, 5, 0, 5, 2, -1, 1, 0, 3]]"
2,2023-01-01,10,"[[-2, 0, 5, -3, 4, -1, 0, 1, 0, 3]]"
3,2023-01-01,14,"[[-3, 2, 0, 1, -1, 0, 0, -4, 0, 5]]"
4,2023-01-01,15,"[[3, -1, 2, 5, 4, 0, 1, 2, 5, 6], [-4, 3, 1, 0..."
...,...,...,...
5565,2023-12-30,14,"[[3, 1, -4, 2, 5, -6, 0, -9, 1], [-4, 0, 2, 1,..."
5566,2023-12-30,15,"[[-3, 0, 2, 1, 4, 5, 0, -1, 0, 3], [-4, 3, 0, ..."
5567,2023-12-30,16,"[[-1, 0, 2, 4, 3, 1, 0, -1, 0, 1]]"
5568,2023-12-30,17,"[[3, 0, 2, -1, 1, -4, 5, 4, -2, 2], [-2, 3, 1,..."


In [61]:
# 3) expand those lists into separate columns processed_0, processed_1, ...
indicators_expanded = pd.DataFrame(
    df_inds_lists["scores_list"].tolist()
).rename(columns=lambda i: f"scores_{i}")

In [62]:
indicators_expanded

,scores_0,scores_1,scores_2,scores_3,scores_4,scores_5,scores_6,scores_7,scores_8,scores_9,...,scores_13,scores_14,scores_15,scores_16,scores_17,scores_18,scores_19,scores_20,scores_21,scores_22
0,"[2, 3, 0, 4, 1, 5, 2, 3, 0, 0]",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,"[-1, 4, 5, 0, 5, 2, -1, 1, 0, 3]",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,"[-2, 0, 5, -3, 4, -1, 0, 1, 0, 3]",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,"[-3, 2, 0, 1, -1, 0, 0, -4, 0, 5]",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,"[3, -1, 2, 5, 4, 0, 1, 2, 5, 6]","[-4, 3, 1, 0, -2, 5, -3, -1, -2, 0]",None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5565,"[3, 1, -4, 2, 5, -6, 0, -9, 1]","[-4, 0, 2, 1, 5, 0, 3, 4, 0, 2]","[-1, -2, 0, -3, 0, -1, 0, -1, 0, -1]","[-1, 2, 3, 4, 5, 0, -1, 5, 0, -1]","[3, -1, 4, 0, 2, 5, 2, 1, -4, 0]","[3, -1, 2, 0, 1, 3, 5, 4, 2, 0]","[-4, 3, 0, 2, 1, 0, 1, -2, 5, 1]","[3, -1, 5, 0, 2, -4, 0, 4, 1, 1]",None,None,...,None,None,None,None,None,None,None,None,None,None
5566,"[-3, 0, 2, 1, 4, 5, 0, -1, 0, 3]","[-4, 3, 0, 5, 1, 2, -1, 0, 0, 0]","[-3, 2, 0, -1, 0, 5, -1, -2, 0, 0]","[-3, -2, 0, 1, 2, 4, 5, 4, 0, 2]",None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5567,"[-1, 0, 2, 4, 3, 1, 0, -1, 0, 1]",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5568,"[3, 0, 2, -1, 1, -4, 5, 4, -2, 2]","[-2, 3, 1, 4, 0, -1, 5, -3, 1, 0]","[-3, -1, 2, 0, 5, -2, -1, 0, -1, 1]",None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [63]:
df_inds_wide = pd.concat([df_inds_lists[["day", "hour"]].reset_index(drop=True),
                           indicators_expanded.reset_index(drop=True)],
                          axis=1)

In [64]:
df_inds_wide

,day,hour,scores_0,scores_1,scores_2,scores_3,scores_4,scores_5,scores_6,scores_7,...,scores_13,scores_14,scores_15,scores_16,scores_17,scores_18,scores_19,scores_20,scores_21,scores_22
0,2023-01-01,6,"[2, 3, 0, 4, 1, 5, 2, 3, 0, 0]",None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,2023-01-01,9,"[-1, 4, 5, 0, 5, 2, -1, 1, 0, 3]",None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2023-01-01,10,"[-2, 0, 5, -3, 4, -1, 0, 1, 0, 3]",None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,2023-01-01,14,"[-3, 2, 0, 1, -1, 0, 0, -4, 0, 5]",None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,2023-01-01,15,"[3, -1, 2, 5, 4, 0, 1, 2, 5, 6]","[-4, 3, 1, 0, -2, 5, -3, -1, -2, 0]",None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5565,2023-12-30,14,"[3, 1, -4, 2, 5, -6, 0, -9, 1]","[-4, 0, 2, 1, 5, 0, 3, 4, 0, 2]","[-1, -2, 0, -3, 0, -1, 0, -1, 0, -1]","[-1, 2, 3, 4, 5, 0, -1, 5, 0, -1]","[3, -1, 4, 0, 2, 5, 2, 1, -4, 0]","[3, -1, 2, 0, 1, 3, 5, 4, 2, 0]","[-4, 3, 0, 2, 1, 0, 1, -2, 5, 1]","[3, -1, 5, 0, 2, -4, 0, 4, 1, 1]",...,None,None,None,None,None,None,None,None,None,None
5566,2023-12-30,15,"[-3, 0, 2, 1, 4, 5, 0, -1, 0, 3]","[-4, 3, 0, 5, 1, 2, -1, 0, 0, 0]","[-3, 2, 0, -1, 0, 5, -1, -2, 0, 0]","[-3, -2, 0, 1, 2, 4, 5, 4, 0, 2]",None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5567,2023-12-30,16,"[-1, 0, 2, 4, 3, 1, 0, -1, 0, 1]",None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
5568,2023-12-30,17,"[3, 0, 2, -1, 1, -4, 5, 4, -2, 2]","[-2, 3, 1, 4, 0, -1, 5, -3, 1, 0]","[-3, -1, 2, 0, 5, -2, -1, 0, -1, 1]",None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [66]:
df = df_stocks.merge(df_inds_wide, on=["day", "hour"], how="left")


In [68]:
df.columns

Index(['timestamp', 'symbol', 'open', 'high', 'low', 'close', 'volume',
       'sma_10', 'sma_50', 'ema_20', 'rsi_14', 'macd', 'macd_signal',
       'macd_hist', 'volatility_20', 'target_up_next', 'day', 'hour',
       'scores_0', 'scores_1', 'scores_2', 'scores_3', 'scores_4', 'scores_5',
       'scores_6', 'scores_7', 'scores_8', 'scores_9', 'scores_10',
       'scores_11', 'scores_12', 'scores_13', 'scores_14', 'scores_15',
       'scores_16', 'scores_17', 'scores_18', 'scores_19', 'scores_20',
       'scores_21', 'scores_22'],
      dtype='object')

In [81]:
df.to_csv("2023_tesla_stocks_news_mixed.csv")

### Trying aggregate scores combinations

In [74]:
score_cols = [f"scores_{i}" for i in range(23)]
def normalize_list(lst):
    lst = np.array(lst, dtype=float)
    max_val = lst.max()
    min_val = lst.min()
    def norm(x):
        if x > 0:
            return x / max_val if max_val != 0 else 0
        elif x < 0:
            return x / abs(min_val) if min_val != 0 else 0
        else:
            return 0
    return np.array([norm(x) for x in lst])

def combine_and_normalize(row):
    lists = [np.array(row[col], dtype=float) for col in score_cols if isinstance(row[col], list)]
    if not lists:
        return np.zeros(10)
    summed = np.sum(lists, axis=0)
    return normalize_list(summed)

In [75]:
df["normalized_list"] = df.apply(combine_and_normalize, axis=1)
df["avg_normalized"] = df["normalized_list"].apply(np.mean)
df["max_normalized"] = df["normalized_list"].apply(np.max)
df["min_normalized"] = df["normalized_list"].apply(np.min)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (7,) + inhomogeneous part.

In [79]:
df.size

42517

In [78]:
for col in score_cols:
    wrong_len = df[col].apply(lambda x: isinstance(x, list) and len(x) != 10)
    if wrong_len.any():
        print(f"{col} has lists with length != 10")
        print (len(wrong_len))

scores_0 has lists with length != 10
1037
scores_1 has lists with length != 10
1037
scores_2 has lists with length != 10
1037
scores_3 has lists with length != 10
1037
scores_4 has lists with length != 10
1037
scores_5 has lists with length != 10
1037
scores_6 has lists with length != 10
1037
scores_7 has lists with length != 10
1037
scores_8 has lists with length != 10
1037
scores_9 has lists with length != 10
1037
scores_10 has lists with length != 10
1037
scores_11 has lists with length != 10
1037
scores_12 has lists with length != 10
1037
scores_14 has lists with length != 10
1037
